# Data Source Investigation

Explore source sample artifacts produced by `python ingestion/minimal_ingest.py` and export summary tables to CSV.


In [ ]:
from pathlib import Path
import json
import zipfile

import pandas as pd


In [ ]:
SAMPLES_DIR = Path('../../ingestion/tmp/samples')
REPORT_PATH = Path('../../ingestion/tmp/samples/source_audit/minimal_ingestion_report.json')
EXPORT_DIR = Path('../../exploration/tmp/notebook_exports/01_data_source_investigation')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if not SAMPLES_DIR.exists():
    raise FileNotFoundError(f'Missing samples dir: {SAMPLES_DIR.resolve()}')

all_files = sorted([p for p in SAMPLES_DIR.rglob('*') if p.is_file()])
print('Samples dir:', SAMPLES_DIR.resolve())
print('Sample files:', len(all_files))
print('Export dir:', EXPORT_DIR.resolve())


In [ ]:
rows = []
for p in all_files:
    rows.append({
        'source': p.parent.name,
        'file': p.name,
        'path': str(p),
        'bytes': p.stat().st_size,
    })
summary_df = pd.DataFrame(rows).sort_values(['source', 'file']).reset_index(drop=True)
summary_df.to_csv(EXPORT_DIR / 'sample_file_inventory.csv', index=False)
summary_df


In [ ]:
if REPORT_PATH.exists():
    report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
    report_df = pd.DataFrame([{
        'name': r.get('name'),
        'ok': r.get('ok'),
        'error': r.get('error')
    } for r in report.get('results', [])])
    report_df.to_csv(EXPORT_DIR / 'probe_status.csv', index=False)
    report_df
else:
    print('No report found at', REPORT_PATH)


## Sample Preview Helpers


In [ ]:
def preview_text(path: Path, n_chars: int = 1200):
    text = path.read_text(encoding='utf-8', errors='replace')
    print(text[:n_chars])

def preview_csv(path: Path, n_rows: int = 5):
    return pd.read_csv(path).head(n_rows)

def preview_zip(path: Path):
    try:
        with zipfile.ZipFile(path, 'r') as zf:
            names = zf.namelist()
        return pd.DataFrame({'zip_member': names})
    except zipfile.BadZipFile:
        raw = path.read_bytes()
        text_head = raw[:240].decode('utf-8', errors='replace').replace('\n', ' ')
        return pd.DataFrame([
            {
                'zip_member': '<not_a_zip>',
                'bytes': len(raw),
                'first_16_hex': raw[:16].hex(),
                'text_head': text_head,
            }
        ])

def preview_geojson(path: Path, n_features: int = 3):
    payload = json.loads(path.read_text(encoding='utf-8'))
    feats = payload.get('features', [])
    rows = []
    for f in feats[:n_features]:
        props = f.get('properties', {})
        rows.append({
            'Code': props.get('Code'),
            'StationName': props.get('StationName'),
            'City': props.get('City'),
            'State': props.get('State'),
            'StnType': props.get('StnType'),
        })
    return pd.DataFrame(rows)


## Source-by-Source Sample Exploration


In [ ]:
fred_preview = preview_csv(SAMPLES_DIR / 'fred' / 'philly_unemployment_head.csv')
fred_preview.to_csv(EXPORT_DIR / 'fred_preview.csv', index=False)
fred_preview


In [ ]:
zillow_preview = preview_csv(SAMPLES_DIR / 'zillow' / 'zori_head.csv')
zillow_preview.to_csv(EXPORT_DIR / 'zillow_preview.csv', index=False)
zillow_preview


In [ ]:
amtrak_preview = preview_geojson(SAMPLES_DIR / 'ntad_amtrak' / 'amtrak_sample.geojson')
amtrak_preview.to_csv(EXPORT_DIR / 'ntad_amtrak_preview.csv', index=False)
amtrak_preview


In [ ]:
gtfs_zip_path = SAMPLES_DIR / 'septa_gtfs' / 'gtfs_public.zip'
if not gtfs_zip_path.exists():
    gtfs_zip_path = SAMPLES_DIR / 'septa_gtfs' / 'gtfs_public_head.zip'
septa_zip_members = preview_zip(gtfs_zip_path)
septa_zip_members.to_csv(EXPORT_DIR / 'septa_gtfs_zip_members.csv', index=False)
septa_zip_members


In [ ]:
preview_text(SAMPLES_DIR / 'septa_gtfs_rt' / 'gtfs_rt_metadata_sample.html', n_chars=1000)


In [ ]:
preview_text(SAMPLES_DIR / 'phl_property_bulk' / 'property_download_page.html', n_chars=1000)


In [ ]:
preview_text(SAMPLES_DIR / 'opendataphilly_rental_suitability' / 'dataset_page.html', n_chars=1000)


In [ ]:
preview_text(SAMPLES_DIR / 'opendataphilly_li_property_history' / 'dataset_page.html', n_chars=1000)
